# **__________   KAYAK PROJECT   ___________**

**The goal of this project is to identify the possible destinations for holidays using APIs and webscrapping to find the best hotels in the cities where the weather forecast suits our wishes best.**

## **I. Identification of the possible destinations according to the weather forecast**

Here our goal will be to identify the 5 best cities for us according to the weather forecast for the 7 following days among a list of 35 most touristic cities in France. The weather API we will use (OpenWeatherMap) needs to be provided with longitude and latitude of each city in order to give us the weather forecast. So we will start to gather these information with another API called Nominatum.

### **1. List of the 35 most touristic cities in France**

One Week In.com identified the top-35 cities to visit in France. We store them into the variable `list_35_cities`.

In [33]:
list_35_cities=["Mont Saint Michel",
"St Malo",
"Bayeux",
"Le Havre",
"Rouen",
"Paris",
"Amiens",
"Lille",
"Strasbourg",
"Chateau du Haut Koenigsbourg",
"Colmar",
"Eguisheim",
"Besancon",
"Dijon",
"Annecy",
"Grenoble",
"Lyon",
"Gorges du Verdon",
"Bormes les Mimosas",
"Cassis",
"Marseille",
"Aix en Provence",
"Avignon",
"Uzes",
"Nimes",
"Aigues Mortes",
"Saintes Maries de la mer",
"Collioure",
"Carcassonne",
"Ariege",
"Toulouse",
"Montauban",
"Biarritz",
"Bayonne",
"La Rochelle"]

### **2. Coordinates of every cities**

We are going to use the information of the **API Nominatim** to get the coordinates of each of the top-35 best cities in France

**2.0. Modules import**

In [4]:
import requests
import pandas as pd
import time

**2.1. API request test on one city : Paris**

To identify the correct parameters for our API request, we can first try a request on a single city and then test it in a loop for all the cities in our list. Let's take Paris for this first exemple.

_2.1.1. Identification of the parameters for the API request_

In the API documentation, we can read that :
- there are different ways to find information, for instance, we can find the location of a city by giving it's name with the "/search" element, or we can do the opposite with "/reverse". Here we need to use the "/search" option.
- the endpoint for a request is "https://nominatim.openstreetmap.org/search?<params>"
- we don't need and authentification
- the parameters we need are `q` for "query", it means the city, the `format` (here we want a json format) and the `addressedetail` to breakdown the adress into elements.

_2.1.2. Creation of the request_

In [35]:

# ------------------------------------- Identify the base URL --------------------------------------

url_base="https://nominatim.openstreetmap.org/search"


# -------------------------- Headers (required to avoid 403 error) ---------------------------------

headers={
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/118.0"
}


# ----------------------------------------- Parameters ---------------------------------------------

params={
    "q":"paris", # In this "query" parameter we select Paris for the fist test 
    "format":"json",
    "addressdetails":"1", # When set to 1, include a breakdown of the address into elements. 
    "limit":"1" # We want only one element per city
}


# --------------------------------------- GET request ----------------------------------------------

response=requests.get(url=url_base,headers=headers,params=params)

# Status analyse
print("Status code:", response.status_code)


# --------------------------------------- JSON display ---------------------------------------------

if response.status_code == 200:
    data_paris_coord = response.json()
    print(data_paris_coord)
else:
    print("Error:", response.text)


# ------------------------------- Get the coordinates of the city ----------------------------------

city_paris="Paris"
lat_paris=data_paris_coord[0]["lat"]
lon_paris=data_paris_coord[0]["lon"]
print(f"\nCoordinates of {city_paris} : (Lat : {lat_paris}, Lon : {lon_paris})")


#--------------------------------- Load the data into a dataframe ---------------------------------

# Creation of a dictionary that represents the row of the DataFrame
dict_paris_coord={
    "City" : city_paris,
    "Latitude" : lat_paris,
    "Longitude" : lon_paris
}

# Creation of the dataframe
df_paris_coord=pd.DataFrame([dict_paris_coord])
display(df_paris_coord)

Status code: 200
[{'place_id': 88715228, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'relation', 'osm_id': 7444, 'lat': '48.8588897', 'lon': '2.3200410', 'class': 'boundary', 'type': 'administrative', 'place_rank': 15, 'importance': 0.897098092136026, 'addresstype': 'suburb', 'name': 'Paris', 'display_name': 'Paris, Île-de-France, France métropolitaine, France', 'address': {'suburb': 'Paris', 'city_district': 'Paris', 'city': 'Paris', 'ISO3166-2-lvl6': 'FR-75C', 'state': 'Île-de-France', 'ISO3166-2-lvl4': 'FR-IDF', 'region': 'France métropolitaine', 'country': 'France', 'country_code': 'fr'}, 'boundingbox': ['48.8155755', '48.9021560', '2.2241220', '2.4697602']}]

Coordinates of Paris : (Lat : 48.8588897, Lon : 2.3200410)


,City,Latitude,Longitude
0,Paris,48.8588897,2.3200410


**2.2. API request for all the cities**

In [36]:

# ------------------------------------- Identify the base URL --------------------------------------

url_base="https://nominatim.openstreetmap.org/search"


# -------------------------- Headers (required to avoid 403 error) ---------------------------------

headers={
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/118.0"
}

# ------------------------------- Loop on the cities in the querry ---------------------------------

list_dict_coord_35_cities=[] # List that will be appended with one dictionary per city

for city in list_35_cities:
    params={
        "q":city, # we loop on the cities of our list 
        "format":"json",
        "addressdetails":"1",
        "limit":"1" 
    }
    response=requests.get(url=url_base,headers=headers,params=params)
    time.sleep(1)

    if response.status_code == 200:
        data = response.json()
    else:
        print("Error:", response.text)


    # Creation of one dictionary per city
    dict_coordinates_35_cities={
        "City" : city,
        "Latitude" : data[0]["lat"],
        "Longitude" : data[0]["lon"]
    }

    # Append each dictionary in the list of dictionaries
    list_dict_coord_35_cities.append(dict_coordinates_35_cities)

**2.3. Creation of a dataframe and export of the data**

In [49]:
# Creation of the dataframe with all the cities' information
df_coord_35_cities=pd.DataFrame(list_dict_coord_35_cities)
display(df_coord_35_cities)

# Export of the dataframe as a CSV file
df_coord_35_cities.to_csv("coordinates_35_cities_csv",index=False)

,City,Latitude,Longitude
0,Mont Saint Michel,48.6359541,-1.5114600
1,St Malo,49.3146950,-96.9538228
2,Bayeux,49.2764624,-0.7024738
3,Le Havre,49.4938975,0.1079732
4,Rouen,49.4404591,1.0939658
5,Paris,48.8588897,2.3200410
6,Amiens,49.8941708,2.2956951
7,Lille,50.6365654,3.0635282
8,Strasbourg,48.5846140,7.7507127
9,Chateau du Haut Koenigsbourg,48.2493820,7.3439412


### **3. Weather forecast for every city**

Now that we have the coordinates of each city, we can access the weather forecast **API Openweathermap** to get the weather forecast for the next 7 days and for each of the cities in our list. First we need to register on the website to get our API key.

**3.0. Modules import**

In [38]:
from dotenv import load_dotenv
import os 

**3.1. API request test on one city : Paris**

_3.1.1. Identification of the parameters for the API request_

The One Call API 2.5 provides the following weather data for any geographical coordinates (longitude/latitude) that we will provide :

- Current weather
- Minute forecast for 1 hour
- Hourly forecast for 48 hours
- Daily forecast for 7 days
- National weather alerts
- Historical weather data for the previous 5 days
- Current and forecast weat

=> We are interested in the Daily forecast for 7 days so we will select the parameter `daily` by excluding the list `current`,`minutely`,`hourly`,`alerts` in the "exclude" parameter

For the units, we will select the parameter `metric` in order to have the temperature in Celsius and the wind speed in metre/sec (otherwise the default temperature is in Kelvin).

Finally, we will have to use an API key so we will store it in our local environment and use the dotenv library to access it.

_3.1.2. Creation of the request_

In [39]:
!pip install python-dotenv

In [40]:
#------------------------------- Load variables from .env file -----------------------------------

load_dotenv()

# -------------------------- Access the key securely from the environment -------------------------

API_KEY = os.getenv("API_KEY")

# -------------------------------------------- API call -------------------------------------------

url_base="https://api.openweathermap.org/data/2.5/forecast"

params = {
    "lat":lon_paris,
    "lon": lat_paris,
    "units":"metric",
    "exclude":"current,minutely,hourly,alerts",
    "appid":API_KEY,
}

response = requests.get(url_base,params=params)
print(response.url)

# -------------------------------------- Response analysis ----------------------------------------

if response.status_code == 200:
    json_paris_weather = response.json()
    print(json_paris_weather)
else:
    print("Error:", response.text)

https://api.openweathermap.org/data/2.5/forecast?lat=2.3200410&lon=48.8588897&units=metric&exclude=current%2Cminutely%2Chourly%2Calerts&appid=28089930b22b3f568e2d5c225cd39da2
{'cod': '200', 'message': 0, 'cnt': 40, 'list': [{'dt': 1776092400, 'main': {'temp': 28.73, 'feels_like': 31.02, 'temp_min': 28.73, 'temp_max': 28.73, 'pressure': 1010, 'sea_level': 1010, 'grnd_level': 1010, 'humidity': 63, 'temp_kf': 0}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04n'}], 'clouds': {'all': 100}, 'wind': {'speed': 3.12, 'deg': 84, 'gust': 3.3}, 'visibility': 10000, 'pop': 0, 'sys': {'pod': 'n'}, 'dt_txt': '2026-04-13 15:00:00'}, {'dt': 1776103200, 'main': {'temp': 28.68, 'feels_like': 31.08, 'temp_min': 28.59, 'temp_max': 28.68, 'pressure': 1011, 'sea_level': 1011, 'grnd_level': 1012, 'humidity': 64, 'temp_kf': 0.09}, 'weather': [{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04n'}], 'clouds': {'all': 100}, 'wind': {'speed': 3.85, 

In [41]:
#---------------------------------- Analysis of the response and the nested dictionaries --------------------------
import json

json_paris_weather=json_paris_weather['list'] # We keep only the list of dictionaries that we need

print(json.dumps(json_paris_weather, indent=4)) # Formatted format for a better readability


[
    {
        "dt": 1776092400,
        "main": {
            "temp": 28.73,
            "feels_like": 31.02,
            "temp_min": 28.73,
            "temp_max": 28.73,
            "pressure": 1010,
            "sea_level": 1010,
            "grnd_level": 1010,
            "humidity": 63,
            "temp_kf": 0
        },
        "weather": [
            {
                "id": 804,
                "main": "Clouds",
                "description": "overcast clouds",
                "icon": "04n"
            }
        ],
        "clouds": {
            "all": 100
        },
        "wind": {
            "speed": 3.12,
            "deg": 84,
            "gust": 3.3
        },
        "visibility": 10000,
        "pop": 0,
        "sys": {
            "pod": "n"
        },
        "dt_txt": "2026-04-13 15:00:00"
    },
    {
        "dt": 1776103200,
        "main": {
            "temp": 28.68,
            "feels_like": 31.08,
            "temp_min": 28.59,
            "temp_max": 2

In [42]:
#--------------------------------- Load the data into a dataframe ---------------------------------

# Our Json response contains nested dictionaries so we will use json_normalize to flatten it in order 
# to use it to create a dataframe.

RECORD_PATH = "weather" #record_path`: The key inside each dictionary that points to the LIST of items we 
#want to flatten.


META_FIELDS = [
        'dt',
        ["main","temp"],
        ["main","feels_like"],
        ["main","temp_min"],
        ["main","temp_max"],
        ["main","pressure"],
        ["main","sea_level"],
        ["main","grnd_level"],
        ["main","humidity"],
        ["main","temp_kf"],
        ["clouds","all"],
        ["wind","speed"],
        ["wind","deg"],
        ["wind","gust"],
        "visibility",
        "pop",
        ["sys","pop"],
        "dt_txt"
        ] # `meta`: A list of keys from the *parent* dictionary that 
#we want to keep and repeat for each flattened record.

json_normalized_paris_weather = pd.json_normalize(
data=json_paris_weather,
record_path=RECORD_PATH,
meta=META_FIELDS,
errors='ignore'  # This is important for handling missing/empty lists
)

# Creation of the dataframe
display(json_normalized_paris_weather) # We don't do pd.datatframe() because the normalized json is already a dataframe, we just need to display it



,id,main,description,icon,dt,main.temp,main.feels_like,main.temp_min,main.temp_max,main.pressure,...,main.humidity,main.temp_kf,clouds.all,wind.speed,wind.deg,wind.gust,visibility,pop,sys.pop,dt_txt
0,804,Clouds,overcast clouds,04n,1776092400,28.73,31.02,28.73,28.73,1010,...,63,0,100,3.12,84,3.3,10000,0,NaN,2026-04-13 15:00:00
1,804,Clouds,overcast clouds,04n,1776103200,28.68,31.08,28.59,28.68,1011,...,64,0.09,100,3.85,91,4.13,10000,0,NaN,2026-04-13 18:00:00
2,804,Clouds,overcast clouds,04n,1776114000,28.5,30.76,28.39,28.5,1011,...,64,0.11,100,3.55,86,3.8,10000,0,NaN,2026-04-13 21:00:00
3,803,Clouds,broken clouds,04n,1776124800,28.13,30.25,28.13,28.13,1010,...,65,0,72,4.03,91,4.26,10000,0,NaN,2026-04-14 00:00:00
4,801,Clouds,few clouds,02d,1776135600,28.27,30.77,28.27,28.27,1011,...,67,0,13,5.03,92,5.26,10000,0,NaN,2026-04-14 03:00:00
5,801,Clouds,few clouds,02d,1776146400,28.43,30.92,28.43,28.43,1013,...,66,0,14,5.22,93,5.52,10000,0,NaN,2026-04-14 06:00:00
6,804,Clouds,overcast clouds,04d,1776157200,28.57,30.89,28.57,28.57,1011,...,64,0,100,4.34,96,4.57,10000,0,NaN,2026-04-14 09:00:00
7,804,Clouds,overcast clouds,04d,1776168000,28.7,30.97,28.7,28.7,1009,...,63,0,99,3.96,97,4.2,10000,0,NaN,2026-04-14 12:00:00
8,803,Clouds,broken clouds,04n,1776178800,28.71,30.99,28.71,28.71,1010,...,63,0,77,3.98,101,4.23,10000,0,NaN,2026-04-14 15:00:00
9,803,Clouds,broken clouds,04n,1776189600,28.63,30.99,28.63,28.63,1012,...,64,0,59,3.99,104,4.24,10000,0,NaN,2026-04-14 18:00:00


**List of variable in the json response and the data frame that we wish to keep (checked variables):**

Variables not nested:
- [ ] **`dt`** : Time of data forecasted, unix, UTC
- [x] **`visibility`**: Average visibility (meter) 
- [x] **`pop`**: Probability of precipitation (0 to 1 where 0=0% and 1=100%) 
- [x] **`dt_txt`** #  Time of data forecasted, ISO, UTC 


Variables in **`main`** : 
- [x] **`temp`** : Temperature (°C) 
- [x] **`feels_like`**: Human perception of the temperature (°C) 
- [x] **`temp_min`**: Minimum temperature at the moment of calculation (°C) . This is minimal forecasted temperature (within large megalopolises and urban areas), use this parameter optionally.
- [x] **`temp_max`**: Maximum temperature at the moment of calculation (°C).  This is minimal forecasted temperature (within large megalopolises and urban areas), use this parameter optionally.
- [ ] **`pressure`**: Atmospheric pressure on the sea level (hPa)
- [ ] **`sea_level`**: Atmospheric pressure on the sea level (hPa)
- [ ] **`grnd_level`**: Atmospheric pressure on the ground level (hPa)
- [x] **`humidity`**: Humidity % 
- [ ] **`temp_kf`**: Internal parameter

Variables in **`weather`** : 
- [ ] **`id`**: Weather condition id
- [ ] **`main`**: Group of weather parameters (Rain, Snow, Clouds etc.)
- [x] **`description`**: Weather condition within the group 
- [ ] **`icon`**: Weather icon id
  
Variable in **`cloud`** : 
- [x] **`all`**: Cloudiness, %

Variables in **`wind`**:
- [x] **`speed`**: Wind speed (meter/sec) 
- [ ] **`deg`**: Wind direction (degree)
- [x] **`gust`**: Wind gust (meter/sec) 

Variable in **`sys`** : 
- [x] **`pop`**: Part of the day (n=night, d=day) 


In [43]:
#----------------------------------- Selection of columns of interest -------------------------------------

columns_to_keep=['dt_txt','main.temp','main.feels_like','main.temp_min','main.temp_max','main.humidity','description','clouds.all','pop','sys.pop','wind.speed','wind.gust','visibility']
df_weather_paris= json_normalized_paris_weather[columns_to_keep].copy()

display(df_weather_paris)


,dt_txt,main.temp,main.feels_like,main.temp_min,main.temp_max,main.humidity,description,clouds.all,pop,sys.pop,wind.speed,wind.gust,visibility
0,2026-04-13 15:00:00,28.73,31.02,28.73,28.73,63,overcast clouds,100,0,NaN,3.12,3.3,10000
1,2026-04-13 18:00:00,28.68,31.08,28.59,28.68,64,overcast clouds,100,0,NaN,3.85,4.13,10000
2,2026-04-13 21:00:00,28.5,30.76,28.39,28.5,64,overcast clouds,100,0,NaN,3.55,3.8,10000
3,2026-04-14 00:00:00,28.13,30.25,28.13,28.13,65,broken clouds,72,0,NaN,4.03,4.26,10000
4,2026-04-14 03:00:00,28.27,30.77,28.27,28.27,67,few clouds,13,0,NaN,5.03,5.26,10000
5,2026-04-14 06:00:00,28.43,30.92,28.43,28.43,66,few clouds,14,0,NaN,5.22,5.52,10000
6,2026-04-14 09:00:00,28.57,30.89,28.57,28.57,64,overcast clouds,100,0,NaN,4.34,4.57,10000
7,2026-04-14 12:00:00,28.7,30.97,28.7,28.7,63,overcast clouds,99,0,NaN,3.96,4.2,10000
8,2026-04-14 15:00:00,28.71,30.99,28.71,28.71,63,broken clouds,77,0,NaN,3.98,4.23,10000
9,2026-04-14 18:00:00,28.63,30.99,28.63,28.63,64,broken clouds,59,0,NaN,3.99,4.24,10000


**3.2 Function for API calls and data preparation**

To call the weather API for each city, we are going to create a function that will request the API and save all the relevant information we have previously identified.

In [105]:
def fetch_weather_data(cities_df):
    """Fetch 7-day weather forecast for each city."""
    
    load_dotenv()
    API_KEY = os.getenv("API_KEY")
    
    if not API_KEY:
        raise ValueError("API_KEY not found in .env file")
    
    url_base = "https://api.openweathermap.org/data/2.5/forecast"
    results = {}
    
    for _, row in cities_df.iterrows():
        city = row['City']
        
        params = {
            "lat": row['Latitude'],
            "lon": row['Longitude'],
            "units": "metric",
            "appid": API_KEY
        }
        
        try:
            response = requests.get(url_base, params=params, timeout=10)
            
            if response.status_code != 200:
                results[city] = (pd.DataFrame(), False)
                continue
            
            # Parse and flatten JSON
            forecast_list = response.json()['list']
            df = pd.json_normalize(forecast_list, errors='ignore')
            
            # Extract weather description
            df['description'] = df['weather'].apply(
                lambda x: x[0]['description'] if isinstance(x, list) and len(x) > 0 else None
            )
            
            # Select and rename columns
            cols_to_keep = {
                'main.temp': 'temp',
                'main.temp_min': 'temp_min',
                'main.temp_max': 'temp_max',
                'main.humidity': 'humidity',
                'pop': 'rain_probability',
                'clouds.all': 'cloudiness',
                'wind.speed': 'wind_speed'
            }
            
            existing_cols = [col for col in cols_to_keep.keys() if col in df.columns]
            weather_df = df[['dt_txt', 'description'] + existing_cols].rename(columns=cols_to_keep)
            
            # Convert types
            weather_df['dt_txt'] = pd.to_datetime(weather_df['dt_txt'])
            for col in weather_df.columns:
                if col not in ['dt_txt', 'description']:
                    weather_df[col] = pd.to_numeric(weather_df[col], errors='coerce')        
            results[city] = (weather_df, True)
            
        except Exception as e:
            results[city] = (pd.DataFrame(), False)
    
    return results

**3.3 Weather API call for all cities**

Now we can use the created function to extract the weather information of the weather forecast for our list of 35 cities.

In [106]:
weather_all_cities_dict = fetch_weather_data(df_coord_35_cities)

In [107]:
weather_all_cities_dict

{'Mont Saint Michel': (                dt_txt       description   temp  temp_min  temp_max  humidity  \
  0  2026-04-14 15:00:00   overcast clouds  13.01     13.01     13.01        87   
  1  2026-04-14 18:00:00   overcast clouds  13.17     13.17     13.49        88   
  2  2026-04-14 21:00:00   overcast clouds  11.72     11.08     11.72        91   
  3  2026-04-15 00:00:00   overcast clouds  11.35     11.35     11.35        93   
  4  2026-04-15 03:00:00   overcast clouds  11.04     11.04     11.04        95   
  5  2026-04-15 06:00:00        light rain  10.92     10.92     10.92        98   
  6  2026-04-15 09:00:00        light rain  13.83     13.83     13.83        92   
  7  2026-04-15 12:00:00        light rain  17.26     17.26     17.26        67   
  8  2026-04-15 15:00:00        light rain  15.82     15.82     15.82        80   
  9  2026-04-15 18:00:00        light rain  14.09     14.09     14.09        81   
  10 2026-04-15 21:00:00         clear sky   9.84      9.84      9

**3.3 Weather preferences**

For most people, a comfortable weather for holidays is : 
- hot but not too hot (around 25°C), 
- with low humidity, wind and clouds. 
- low rain probability

We are going to create a scoring functions with these parameters to determine the weather score of a given city. Then will be able to compare cities and select the best ones.

In [108]:
 
def score_city_weather(weather_df, ideal_temp=25):
    """Score a city's weather based on 7-day forecast."""
    
    # Aggregate by day
    daily = weather_df.groupby(weather_df['dt_txt'].dt.date).agg({
        'temp': 'mean',
        'temp_min': 'min',
        'temp_max': 'max',
        'humidity': 'mean',
        'rain_probability': 'mean'
    })
    
    # Calculate component scores [0, 1]
    temp_score = (1 - abs(daily['temp'] - ideal_temp) / 10).clip(0, 1)
    humidity_score = 1 - (daily['humidity'] / 100)
    rain_score = 1 - daily['rain_probability']
    stability_score = (1 - (daily['temp_max'] - daily['temp_min']) / 10).clip(0, 1)
    
    # Weighted final score
    daily_score = (
        0.4 * temp_score +
        0.2 * humidity_score +
        0.3 * rain_score +
        0.1 * stability_score
    )
    
    # City score: mean minus std (penalize variability)
    city_score = daily_score.mean() - daily_score.std()
    
    return {
        'avg_temp': daily['temp'].mean(),
        'avg_humidity': daily['humidity'].mean(),
        'rain_prob': daily['rain_probability'].mean() * 100,
        'temp_stability': daily['temp'].std(),
        'score': city_score
    }
 
 
def calculate_weather_scores(weather_results):
    """Calculate scores for all cities."""
    
    scores = []
    
    for city, (weather_df, success) in weather_results.items():
        if not success or weather_df.empty:
            continue
        
        stats = score_city_weather(weather_df)
        scores.append({
            'City': city,
            'Weather_Score': round(stats['score'], 3),
            'Avg_Temp': round(stats['avg_temp'], 1),
            'Avg_Humidity': round(stats['avg_humidity'], 1),
            'Rain_Probability': round(stats['rain_prob'], 1),
            'Temp_Stability': round(stats['temp_stability'], 2)
        })
    
    df = pd.DataFrame(scores).sort_values('Weather_Score', ascending=False).reset_index(drop=True)
    return df


**3.4. Weather scoring for all cities**

In [ ]:
weather_score_all_df = calculate_weather_scores(weather_all_cities_dict)
weather_score_all_df

,City,Weather_Score,Avg_Temp,Avg_Humidity,Rain_Probability,Temp_Stability,Success
0,Collioure,0.453,16.6,56.0,0.0,2.17,True
1,Saintes Maries de la mer,0.412,15.0,64.3,0.0,2.36,True
2,Aix en Provence,0.403,16.0,56.3,0.0,3.25,True
3,Aigues Mortes,0.401,15.4,60.9,0.0,2.26,True
4,Marseille,0.395,15.9,63.3,1.1,2.69,True
5,Nimes,0.394,15.0,59.1,0.0,3.15,True
6,Avignon,0.392,14.7,63.3,0.0,3.32,True
7,Cassis,0.390,15.3,63.6,1.4,2.12,True
8,Uzes,0.381,13.8,64.0,0.0,2.84,True
9,Strasbourg,0.362,12.2,74.4,3.0,1.90,True


**3.5. Export of the data**

In [63]:
weather_score_all_df.to_csv("weather_best_destinations_csv",index=True)

Top five destinations according to our weather scoring :

In [74]:
top_five_destinations_df = weather_score_all_df.head(5).copy()
top_five_cities_names = weather_score_all_df.head(5)["City"].tolist()

In [78]:
print("The best cities according to the actual weather forecast are :\n")
for name, i in zip(top_five_cities_names,range(1,6)) : 
    print(f"  - N°{i} : {name}")

The best cities according to the actual weather forecast are :

  - N°1 : Collioure
  - N°2 : Saintes Maries de la mer
  - N°3 : Aix en Provence
  - N°4 : Aigues Mortes
  - N°5 : Marseille


In [ ]:
top_five_destinations_df.to_csv("DATA/bookingHotels/top_five_destinations_csv.csv", index=True)

## **II. Identification of the 20 best hotels in the top-5 destinations**

Now that we have defined our top-5 list of destinations, let's check the 20 bests hotels per city using webscrapping techniques on the Booking.com website

### **1. Create a webscrapping function**

As Booking.com actively blocks scraping, we are using Selenium here to gather information for each hotel with : 
- its name,
- the url to its booking.com page,
- Its coordinates: latitude and longitude,
- Its score according to the website users,
- A text description of the hotel.

In [13]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
from datetime import datetime, timedelta, date

In [9]:
def scrape_hotels(city, check_in, check_out, num_hotels=20):
    """Scrape hotels for one city from Booking.com."""
    
    options = Options()
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    options.add_argument("--no-sandbox")
    
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    
    hotels = []
    
    try:
        url = f"https://www.booking.com/searchresults.fr.html?ss={city}&checkin={check_in}&checkout={check_out}&nrpersons=2"
        driver.get(url)
        
        # Wait and scroll
        time.sleep(3)
        driver.execute_script("window.scrollBy(0, window.innerHeight);")
        time.sleep(2)
        
        # Extract hotels
        hotel_elements = driver.find_elements(By.CSS_SELECTOR, "div[data-testid='property-card']")
        
        for elem in hotel_elements[:num_hotels]:
            try:
                # Name
                name = elem.find_element(By.CSS_SELECTOR, "div[data-testid='title']").text
                
                # URL
                link = elem.find_element(By.CSS_SELECTOR, "a[href*='hotel']")
                url_hotel = link.get_attribute("href")
                if not url_hotel.startswith("http"):
                    url_hotel = "https://www.booking.com" + url_hotel
                
                # Rating
                rating = None
                try:
                    import re
                    rating_text = elem.text
                    match = re.search(r'(\d+[.,]\d)', rating_text)
                    if match:
                        rating = float(match.group(1).replace(',', '.'))
                except:
                    pass
                
                hotels.append({
                    'name': name,
                    'url': url_hotel,
                    'rating': rating,
                    'city': city
                })
            except:
                continue
    
    except Exception as e:
        print(f"Error scraping {city}: {str(e)}")
    
    finally:
        driver.quit()
    
    return hotels

### **2. Get hotels information**

In [16]:
def scrape_all_hotels(cities, check_in=None, check_out=None, num_hotels=20):
    """Scrape hotels for all cities."""
    
    if check_in is None:
        check_in = date.today()
        
    if check_out is None:
        check_out = check_in + timedelta(days=7)

    print(check_in, check_out)
    all_hotels = []
    
    for city in cities:
        hotels = scrape_hotels(city, check_in, check_out, num_hotels)
        all_hotels.extend(hotels)
    
    if not all_hotels:
        print("No hotels scraped!")
        return pd.DataFrame()
    
    df = pd.DataFrame(all_hotels)
    df = df.sort_values(['city', 'rating'], ascending=[True, False]).reset_index(drop=True)
    
    return df

In [11]:
cities = pd.read_csv("DATA/bookingHotels/top_five_destinations_csv.csv")

In [17]:
hotels_df = scrape_all_hotels(cities)

2026-04-24 2026-05-01


In [43]:
hotels_df.head()

,name,url,rating,city
0,Casa Gin Tazones,https://www.booking.com/hotel/es/casa-gin-tazo...,9.8,Avg_Humidity
1,Casa Tirador,https://www.booking.com/hotel/es/casa-tirador....,9.8,Avg_Humidity
2,El Rincón de Silvia,https://www.booking.com/hotel/es/el-rincon-de-...,9.7,Avg_Humidity
3,Nordés Apartamentos Turísticos - Bañugues,https://www.booking.com/hotel/es/nordes-aparta...,9.5,Avg_Humidity
4,Hotel El Pescador,https://www.booking.com/hotel/es/el-pescador-t...,9.4,Avg_Humidity


In [18]:
# Save to CSV
hotels_df.to_csv("Data/bookingHotels/all_cities_hotels.csv", index=False)


### **3. Visualizations**

**3.1. Top destinations map**

In [38]:
import plotly.express as px

def plot_top_destinations(df):
    """Interactive map with modern Mapbox-style background."""

    top5 = df.head(5)

    fig = px.scatter_mapbox(
        top5,
        lat="Latitude",
        lon="Longitude",
        text="City",
        zoom=4.5,
        center={"lat": 46.6, "lon": 2.5},  # France
        height=600
    )

    fig.update_traces(
        marker=dict(size=10, color="red"),
        hovertemplate="<b>%{text}</b><extra></extra>"
    )

    fig.update_layout(
        mapbox_style="carto-positron",  
        margin={"r":0,"t":40,"l":0,"b":0},
        title="<b>Top 5 Best Destinations in France</b>"
    )

    return fig

In [26]:
coordinates_35_cities = pd.read_csv("DATA/BookingHotels/coordinates_35_cities_csv.csv")

In [30]:
selected_cities = [
    "Collioure",
    "Saintes Maries de la mer",
    "Aix en Provence",
    "Aigues Mortes",
    "Marseille"
]

# filtrer les 5 villes
top_cities_coordinates = coordinates_35_cities[coordinates_35_cities["City"].isin(selected_cities)]

In [31]:
top_cities_coordinates

,City,Latitude,Longitude
20,Marseille,43.296174,5.369953
21,Aix en Provence,43.529842,5.447474
25,Aigues Mortes,43.566152,4.191540
26,Saintes Maries de la mer,43.451592,4.427720
27,Collioure,42.525050,3.083155


In [39]:
plot_top_destinations(top_cities_coordinates)

**3.2. Top hotels map**

In [40]:
def plot_top_hotels(hotels_df):
    """Plot top-20 hotels on interactive map."""
    
    # Remove hotels without coordinates
    hotels_with_coords = hotels_df.dropna(subset=['latitude', 'longitude'])
    
    if len(hotels_with_coords) == 0:
        print("No hotels with coordinates found")
        return None
    
    top20 = hotels_with_coords.nlargest(20, 'rating')
    
    # Color by city
    colors = {'Paris': '#1f77b4', 'Barcelona': '#ff7f0e', 'Miami': '#2ca02c', 
              'Rome': '#d62728', 'Amsterdam': '#9467bd'}
    
    fig = go.Figure()
    
    for city in top20['city'].unique():
        city_hotels = top20[top20['city'] == city]
        
        fig.add_trace(go.Scattergeo(
            lon=city_hotels['longitude'],
            lat=city_hotels['latitude'],
            mode='markers',
            marker=dict(
                size=city_hotels['rating'] * 3,
                color=colors.get(city, '#7f7f7f'),
                opacity=0.8,
                line=dict(width=2, color='white')
            ),
            hovertext=('<b>' + city_hotels['name'] + '</b><br>' +
                      'Rating: ' + city_hotels['rating'].astype(str) + '/10'),
            hoverinfo='text',
            name=city
        ))
    
    fig.update_layout(
        title='<b>Top-20 Hotels by Rating</b>',
        geo=dict(
            projection_type='natural earth',
            showland=True,
            landcolor='rgb(243, 243, 243)',
            showocean=True,
            oceancolor='rgb(204, 229, 255)'
        ),
        height=600
    )
    
    return fig

In [42]:
hotels_df.head()

,name,url,rating,city
0,Casa Gin Tazones,https://www.booking.com/hotel/es/casa-gin-tazo...,9.8,Avg_Humidity
1,Casa Tirador,https://www.booking.com/hotel/es/casa-tirador....,9.8,Avg_Humidity
2,El Rincón de Silvia,https://www.booking.com/hotel/es/el-rincon-de-...,9.7,Avg_Humidity
3,Nordés Apartamentos Turísticos - Bañugues,https://www.booking.com/hotel/es/nordes-aparta...,9.5,Avg_Humidity
4,Hotel El Pescador,https://www.booking.com/hotel/es/el-pescador-t...,9.4,Avg_Humidity


### **4. Save elements to S3** ###

In [ ]:
 
def upload_to_s3(dataframe, filename, bucket_name="kayak_project_bucket"):
    """Upload DataFrame as CSV to S3."""
    
    try:
        # Create S3 client
        s3 = boto3.client('s3')
        
        # Convert to CSV in memory
        csv_buffer = dataframe.to_csv(index=False).encode('utf-8')
        
        # Upload
        s3.put_object(Bucket=bucket_name, Key=filename, Body=csv_buffer)
        logger.info(f"✓ Uploaded {filename} to s3://{bucket_name}/")
        
    except Exception as e:
        logger.error(f"Failed to upload {filename}: {str(e)}")
 
 
def save_figure_to_s3(figure, filename, bucket_name="kayak_project_bucket"):
    """Save Plotly figure as HTML to S3."""
    
    try:
        s3 = boto3.client('s3')
        
        # Save to HTML string
        html_string = figure.to_html()
        
        # Upload
        s3.put_object(
            Bucket=bucket_name,
            Key=filename,
            Body=html_string.encode('utf-8'),
            ContentType='text/html'
        )
        logger.info(f"✓ Uploaded {filename} to s3://{bucket_name}/")
        
    except Exception as e:
        logger.error(f"Failed to upload {filename}: {str(e)}")
 